# Full-Universe Evaluation

Evaluates all 8 prediction methods on the **complete M[t]=0 universe** (no negative subsampling),
separately for observation years **t = 2015** and **t = 2016**.

### Why full universe?
The standard test set uses a 5:1 negative-sampling ratio (~14.5% positive), inflating PR-AUC and
ranking metrics relative to the true deployment setting (~1.7% positive). This notebook evaluates
every (country, product) pair where M[t]=0 to give realistic absolute numbers.

| Subset | Pairs | Positive rate |
|--------|-------|---------------|
| Standard test set (2015, sampled) | 127,531 | 14.5% |
| **Full universe 2015** | **~1,078,236** | **1.71%** |
| **Full universe 2016** | **~1,079,598** | **1.73%** |

### Label definition
- **Positive (1):** M[t]=0 AND M[t+5]=1 AND M[t+6]=1 (sustained new comparative advantage)
- **Negative (0):** M[t]=0 AND M[t+5]=0

### CWR fix
Products missing from the 2010 ubiquity reference now receive **`fillna(ubiq.median())`** instead
of `fillna(0)`, preventing them from being assigned maximum (rarest-product) weight.

### Output
All CSVs are saved to `full_universe_eval/`.

In [30]:
import os, sys, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score, ndcg_score
from torch_geometric.data import HeteroData
from sklearn.decomposition import PCA
from torch_geometric.nn import SAGEConv, GATConv, GCNConv, to_hetero
warnings.filterwarnings('ignore')

DATA_DIR  = 'data'
OUT_DIR   = 'full_universe_eval'
CKPT_DIR  = os.path.join(DATA_DIR, 'models', 'gnn', 'checkpoints')
TRAIN_CUTOFF = 2012
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
PCA_DIM   = 32
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Device: {DEVICE}')
print(f'Output dir: {os.path.abspath(OUT_DIR)}')

# ── Raw data ──────────────────────────────────────────────────────────────────
smooth  = pd.read_csv(os.path.join(DATA_DIR, 'M_cpt_smoothed.csv'))
rca_df  = pd.read_csv(os.path.join(DATA_DIR, 'rca_cpt.csv'))

countries = sorted(smooth['country'].unique())
products  = sorted(smooth['product'].unique())
C, P = len(countries), len(products)
c_idx = {c: i for i, c in enumerate(countries)}
p_idx = {p: i for i, p in enumerate(products)}
print(f'Universe: {C} countries Ã— {P} products = {C*P:,} possible pairs')

def build_M(year):
    M = np.zeros((C, P), dtype=np.float32)
    yr = smooth[smooth['year'] == year]
    ci = yr['country'].map(c_idx).values
    pi = yr['product'].map(p_idx).values
    M[ci, pi] = 1.0
    return M

# ── Fixed PCI proxy — use median for products missing from 2010 reference ─────
rca_ref      = rca_df[rca_df['year'] == 2010]
ubiq         = rca_ref.groupby('product')['rca'].apply(lambda x: (x >= 1).sum())
ubiq_median  = float(ubiq.median())
max_ubiq     = float(ubiq.max())
pci_dict     = {int(p): float(-u / max_ubiq) for p, u in ubiq.items()}
pci_fill_val = -ubiq_median / max_ubiq
print(f'PCI: {len(pci_dict)} products with ubiquity | missing fill = {pci_fill_val:.4f} (was 0.0)')
print(f'Products missing 2010 ubiquity: {sum(1 for p in products if p not in pci_dict)} / {P}')

# ── Proximity matrix (training years only, no leakage) ────────────────────────
print('\nBuilding proximity matrix from training years (≤2012)...')
co_exp  = np.zeros((P, P), dtype=np.float32)
any_exp = np.zeros((P, P), dtype=np.float32)
for yr in sorted(y for y in smooth['year'].unique() if y <= TRAIN_CUTOFF):
    M = build_M(yr)
    co      = M.T @ M
    ex      = M.sum(axis=0)
    any_exp += ex[:, None] + ex[None, :] - co
    co_exp  += co
phi = np.where(any_exp > 0, co_exp / (any_exp + 1e-9), 0.0)
np.fill_diagonal(phi, 0.0)
phi_row_sum = phi.sum(axis=1)
print('Proximity matrix done.')

# ── GNN artifacts ─────────────────────────────────────────────────────────────
edge_idx_raw   = torch.load(os.path.join(DATA_DIR, 'edge_index_by_year.pt'), weights_only=False)
edge_idx_by_yr = {k: v.long() for k, v in edge_idx_raw.items()}
p_x_by_yr      = torch.load(os.path.join(DATA_DIR, 'product_x_by_year.pt'),  weights_only=False)
c_x_11feat     = torch.load(os.path.join(DATA_DIR, 'country_x_by_year.pt'),  weights_only=False)
cap_ei         = torch.load(os.path.join(DATA_DIR, 'capability_edge_index.pt'), weights_only=False).long()

with open(os.path.join(DATA_DIR, 'country_mapping.pkl'), 'rb') as f: c_map = pickle.load(f)
with open(os.path.join(DATA_DIR, 'product_mapping.pkl'), 'rb') as f: p_map = pickle.load(f)

c_feat_df = pd.read_csv(os.path.join(DATA_DIR, 'country_features.csv'))
BACI_COLS = ['log_export', 'n_products', 'avg_rca', 'max_rca']
c_x_4feat = {}
for yr in sorted(c_feat_df['year'].unique()):
    yd = c_feat_df[c_feat_df['year'] == yr].copy()
    yd['idx'] = yd['country'].map(c_map['to_idx'])
    yd = yd.dropna(subset=['idx']).sort_values('idx')
    c_x_4feat[int(yr)] = torch.tensor(yd[BACI_COLS].values, dtype=torch.float32)

# ── LLM embeddings + cosine weights (for v2 model) ───────────────────────────
llm_emb = torch.load(
    os.path.join(DATA_DIR, 'product_llm_embeddings.pt'),
    weights_only=False, map_location='cpu'
).float()  # [5018, 768], unit-normalised

# Augmented product features: [5018, 771] = 3 BACI + 768 LLM
p_x_by_yr_v2 = {yr: torch.cat([base, llm_emb], dim=1)
                for yr, base in p_x_by_yr.items()}

# Cosine weights for every capability edge: [144192, 1]
emb_np = llm_emb.numpy()
cos_weights = torch.tensor(
    (emb_np[cap_ei[0].numpy()] * emb_np[cap_ei[1].numpy()]).sum(axis=1),
    dtype=torch.float32
).unsqueeze(1)

# Scalar cosine weights for GCNConv (Variant B): [144192]
cos_weights_1d = cos_weights.squeeze(1)

# ── PCA-compressed LLM features (Variants A & B) ─────────────────────────────
pca = PCA(n_components=PCA_DIM, random_state=42)
pca.fit(emb_np)
explained = pca.explained_variance_ratio_.sum()
llm_pca_np = pca.transform(emb_np).astype('float32')
llm_pca_np /= (llm_pca_np ** 2).sum(axis=1, keepdims=True) ** 0.5
llm_pca_np = llm_pca_np.clip(-1e8, 1e8)  # safety
llm_pca_t   = torch.from_numpy(llm_pca_np)

P_IN_PCA = 3 + PCA_DIM   # 35
p_x_with_pca = {yr: torch.cat([base, llm_pca_t], dim=1)
                for yr, base in p_x_by_yr.items()}

print('\nAll artifacts loaded.')
print(f'Capability edges: {tuple(cap_ei.shape)}')
print(f'LLM embeddings: {llm_emb.shape}  |  cos weight range [{cos_weights.min():.3f}, {cos_weights.max():.3f}]')
print(f'PCA({PCA_DIM}d) explains {explained*100:.1f}% variance  |  P_IN_PCA={P_IN_PCA}')

Device: cuda
Output dir: c:\Users\Ashwa\Ash_projects\Indigo_Research\full_universe_eval
Universe: 233 countries Ã— 5018 products = 1,169,194 possible pairs
PCI: 4938 products with ubiquity | missing fill = -0.1935 (was 0.0)
Products missing 2010 ubiquity: 80 / 5018

Building proximity matrix from training years (≤2012)...
Proximity matrix done.

All artifacts loaded.
Capability edges: (2, 144192)
LLM embeddings: torch.Size([5018, 768])  |  cos weight range [0.319, 1.000]
PCA(32d) explains 61.6% variance  |  P_IN_PCA=35


## Build Full-Universe Label Sets

For each observation year t, we enumerate **every** (country, product) pair where M[t]=0 and
label it as positive (1) if M[t+5]=1 AND M[t+6]=1, or negative (0) if M[t+5]=0.

In [31]:
def build_full_universe(t):
    """
    Return a DataFrame of all (country, product) pairs where M[t]=0,
    with labels (1 if sustained entry by t+5, 0 if still absent).
    Also adds PCI weight and ECI/density arrays.
    """
    M_t  = build_M(t)
    M_t5 = build_M(t + 5)
    M_t6 = build_M(t + 6)

    # All pairs where M[t]=0
    ci_all, pi_all = np.where(M_t == 0)
    # Only keep pairs that are either clearly positive or clearly negative
    pos_mask = (M_t5[ci_all, pi_all] == 1) & (M_t6[ci_all, pi_all] == 1)
    neg_mask = (M_t5[ci_all, pi_all] == 0)
    keep     = pos_mask | neg_mask

    ci_all = ci_all[keep]
    pi_all = pi_all[keep]
    labels = pos_mask[keep].astype(int)

    df = pd.DataFrame({
        'country': [countries[i] for i in ci_all],
        'product': [products[i] for i in pi_all],
        'ci':      ci_all,
        'pi':      pi_all,
        'label':   labels,
    })

    # Fixed PCI weights
    df['pci'] = df['product'].map(pci_dict).fillna(pci_fill_val)
    min_pci   = df['pci'].min()
    df['w']   = df['pci'] - min_pci

    pos_rate = labels.mean() * 100
    print(f't={t}: {len(df):,} pairs | {labels.sum():,} positives ({pos_rate:.2f}%) | '
          f'{(labels==0).sum():,} negatives')
    return df, M_t

df_2015, M_2015 = build_full_universe(2015)
df_2016, M_2016 = build_full_universe(2016)

t=2015: 1,078,236 pairs | 18,477 positives (1.71%) | 1,059,759 negatives
t=2016: 1,079,598 pairs | 18,714 positives (1.73%) | 1,060,884 negatives


## Metric Definitions

All metrics computed on the **full universe** (no sampling).

| Metric | Definition |
|--------|------------|
| **PR-AUC** | Precision-Recall AUC — primary metric for imbalanced data |
| **AUROC** | ROC AUC |
| **NDCG@20** | Normalized Discounted Cumulative Gain at rank 20, per country (macro avg) |
| **Prec@20** | Precision at rank 20, per country (macro avg) |
| **CWR** | Complexity-Weighted Recall: top-50% predictions, weighted by 1/ubiquity[2010] |
| **Best F1** | F1 at threshold that maximises F1 |
| **P@1000** | Fraction of top-1000 globally scored pairs that are positives |
| **mAP@10** | Mean Average Precision at rank 10, per country (macro avg) |

**CWR note:** Now uses median ubiquity for products missing from 2010 reference (fixes max-weight bug).

In [32]:
def compute_metrics(scores, df, skip_map10=False):
    """
    scores : 1-D array aligned with df rows
    df     : DataFrame with columns label, country, product, w
    """
    scores  = np.asarray(scores, dtype=np.float64)
    labels  = df['label'].values

    # PR-AUC
    prec, rec, _ = precision_recall_curve(labels, scores)
    pr_auc = auc(rec, prec)

    # AUROC
    auroc = roc_auc_score(labels, scores) if 0 < labels.mean() < 1 else 0.0

    # Best F1
    denom  = prec + rec
    f1_all = np.where(denom > 0, 2 * prec * rec / denom, 0.0)
    best_f1 = float(f1_all.max())

    # P@1000
    K = min(1000, len(scores))
    topk = np.argsort(scores)[::-1][:K]
    p1k  = float(labels[topk].sum()) / K

    # Per-country ranking metrics
    tmp = df.copy()
    tmp['score'] = scores
    ndcg_vals, prec20_vals, ap10_vals = [], [], []
    for _, grp in tmp.groupby('country'):
        if grp['label'].sum() == 0:
            continue
        yt = grp['label'].values
        ys = grp['score'].values
        try:
            ndcg_vals.append(ndcg_score([yt], [ys], k=20))
        except Exception:
            pass
        top20 = grp.sort_values('score', ascending=False).head(20)
        prec20_vals.append(top20['label'].mean())
        if not skip_map10:
            n_pos = int(grp['label'].sum())
            top10 = grp.sort_values('score', ascending=False).head(10)['label'].values
            cumtp = np.cumsum(top10)
            prec_k = cumtp / np.arange(1, len(top10) + 1)
            ap = (prec_k * top10).sum() / min(n_pos, 10)
            ap10_vals.append(ap)

    ndcg20 = float(np.nanmean(ndcg_vals))   if ndcg_vals   else 0.0
    prec20 = float(np.nanmean(prec20_vals)) if prec20_vals else 0.0
    map10  = float(np.nanmean(ap10_vals))   if ap10_vals   else float('nan')

    # CWR (fixed: percentile-ranked, median fill for missing products)
    pct   = pd.Series(scores).rank(pct=True).values
    tot_w = df.loc[labels == 1, 'w'].sum()
    hit_w = df.loc[(labels == 1) & (pct >= 0.5), 'w'].sum()
    cwr   = float(hit_w / tot_w) if tot_w > 0 else 0.0

    return {
        'PR-AUC':  round(pr_auc,  4),
        'AUROC':   round(auroc,   4),
        'NDCG@20': round(ndcg20,  4),
        'Prec@20': round(prec20,  4),
        'CWR':     round(cwr,     4),
        'Best F1': round(best_f1, 4),
        'P@1000':  round(p1k,     4),
        'mAP@10':  'N/A' if skip_map10 else round(map10, 4),
    }

# Per-year result stores
RESULTS = {2015: {}, 2016: {}}
# Score cache — used later for RCA > 0.25 filtered evaluation
SCORES  = {2015: {}, 2016: {}}

def evaluate(year, name, scores, skip_map10=False):
    df = df_2015 if year == 2015 else df_2016
    # Cache scores aligned to full-universe df
    SCORES[year][name] = np.asarray(scores, dtype=np.float64)
    res = compute_metrics(scores, df, skip_map10=skip_map10)
    RESULTS[year][name] = res
    m10s = res['mAP@10'] if skip_map10 else f"{res['mAP@10']:.4f}"
    print(f'[{year}] {name:<26}  PR-AUC={res["PR-AUC"]:.4f}  NDCG@20={res["NDCG@20"]:.4f}  '
          f'Prec@20={res["Prec@20"]:.4f}  CWR={res["CWR"]:.4f}  '
          f'BestF1={res["Best F1"]:.4f}  P@1000={res["P@1000"]:.4f}  mAP@10={m10s}')
    return res

print('Metric functions ready.')

Metric functions ready.


## Method 1 — RCA Persistence

Score = fraction of past 3 years where country had RCA ≥ 1 in this product.  
Pure autocorrelation — no learning.

In [33]:
def rca_persistence_scores(t, df):
    hist_yrs = [t - 2, t - 1, t]
    rca_hist = rca_df[rca_df['year'].isin(hist_yrs)][['country', 'product', 'year', 'rca']]
    rca_hist = rca_hist.merge(df[['country', 'product']], on=['country', 'product'])
    rca_wide = rca_hist.pivot_table(index=['country', 'product'], columns='year',
                                    values='rca', fill_value=0)
    for yr in hist_yrs:
        if yr not in rca_wide.columns:
            rca_wide[yr] = 0
    rca_wide['score'] = (rca_wide[hist_yrs] >= 1).mean(axis=1)
    merged = df.merge(rca_wide[['score']], on=['country', 'product'], how='left').fillna(0)
    return merged['score'].values

for yr in [2015, 2016]:
    df_yr = df_2015 if yr == 2015 else df_2016
    scores = rca_persistence_scores(yr, df_yr)
    evaluate(yr, 'RCA Persistence', scores)

[2015] RCA Persistence             PR-AUC=0.2448  NDCG@20=0.1331  Prec@20=0.1369  CWR=0.3359  BestF1=0.1957  P@1000=0.1150  mAP@10=0.0661
[2016] RCA Persistence             PR-AUC=0.2470  NDCG@20=0.1360  Prec@20=0.1367  CWR=0.3349  BestF1=0.1999  P@1000=0.1380  mAP@10=0.0572


## Method 2 — Product Space Density

Score = proximity-weighted fraction of neighboring products the country already exports.  
Proximity matrix computed from training years only (≤2012).

In [34]:
def density_scores(M_t, df):
    dens_mat = (M_t @ phi) / (phi_row_sum[None, :] + 1e-9)
    return dens_mat[df['ci'].values, df['pi'].values]

for yr, M_t, df_yr in [(2015, M_2015, df_2015), (2016, M_2016, df_2016)]:
    evaluate(yr, 'Density', density_scores(M_t, df_yr))

[2015] Density                     PR-AUC=0.0549  NDCG@20=0.1397  Prec@20=0.1274  CWR=0.8865  BestF1=0.1088  P@1000=0.1270  mAP@10=0.0670
[2016] Density                     PR-AUC=0.0596  NDCG@20=0.1323  Prec@20=0.1221  CWR=0.8864  BestF1=0.1141  P@1000=0.1550  mAP@10=0.0642


## Method 3 — ECI

Per-country signal only (all products in a country share the same score).  
mAP@10 is N/A — ECI provides no within-country ranking.

In [35]:
def compute_eci(M_t):
    kc = M_t.sum(axis=1); kp = M_t.sum(axis=0)
    kc_safe = np.where(kc > 0, kc, 1.0)
    kp_safe = np.where(kp > 0, kp, 1.0)
    kc_n, kp_n = kc.astype(float), kp.astype(float)
    for _ in range(20):
        kc_n = (1.0 / kc_safe) * (M_t   @ kp_n)
        kp_n = (1.0 / kp_safe) * (M_t.T @ kc_n)
    return (kc_n - kc_n.mean()) / (kc_n.std() + 1e-9)

for yr, M_t, df_yr in [(2015, M_2015, df_2015), (2016, M_2016, df_2016)]:
    eci = compute_eci(M_t)
    evaluate(yr, 'ECI', eci[df_yr['ci'].values], skip_map10=True)

[2015] ECI                         PR-AUC=0.0165  NDCG@20=0.0195  Prec@20=0.0248  CWR=0.4493  BestF1=0.0348  P@1000=0.0000  mAP@10=N/A
[2016] ECI                         PR-AUC=0.0193  NDCG@20=0.0198  Prec@20=0.0239  CWR=0.4264  BestF1=0.0356  P@1000=0.0090  mAP@10=N/A


## Method 4 — ECI + Density

In [36]:
def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

for yr, M_t, df_yr in [(2015, M_2015, df_2015), (2016, M_2016, df_2016)]:
    eci   = compute_eci(M_t)
    dens  = density_scores(M_t, df_yr)
    combo = minmax(eci[df_yr['ci'].values]) + minmax(dens)
    evaluate(yr, 'ECI + Density', combo)

[2015] ECI + Density               PR-AUC=0.0549  NDCG@20=0.1397  Prec@20=0.1274  CWR=0.8865  BestF1=0.1088  P@1000=0.1270  mAP@10=0.0670
[2016] ECI + Density               PR-AUC=0.0596  NDCG@20=0.1323  Prec@20=0.1221  CWR=0.8864  BestF1=0.1141  P@1000=0.1550  mAP@10=0.0642


## Method 5 — KNN on LLM Embeddings

Score = cosine similarity: target product embedding vs. mean basket embedding of the country's current exports.  
Uses FinLang/finance-embeddings-investopedia (768-dim).

In [57]:
# Load 768-dim embeddings and derive 32-dim PCA version.
EMB_PATH = os.path.join(DATA_DIR, 'product_llm_embeddings.pt')
emb = torch.load(EMB_PATH, weights_only=False, map_location='cpu').numpy()  # [P, 768]
print(f'Embeddings: {emb.shape}')

from sklearn.decomposition import PCA as _PCA
_pca_knn = _PCA(n_components=32, random_state=42)
_pca_knn.fit(emb)
emb_pca = _pca_knn.transform(emb).astype(np.float32)  # [P, 32]
emb_pca /= np.linalg.norm(emb_pca, axis=1, keepdims=True).clip(min=1e-8)
print(f'PCA-LLM: {emb_pca.shape}  '
      f'(explains {_pca_knn.explained_variance_ratio_.sum()*100:.1f}% variance)')

def knn_scores(M_t, df_yr):
    # Vectorised: mean basket in 32-dim PCA space, cosine dot product.
    basket = np.zeros((C, 32), dtype=np.float32)
    for ci_val in range(C):
        exported = np.where(M_t[ci_val] == 1)[0]
        if len(exported):
            b = emb_pca[exported].mean(axis=0)
            norm = np.linalg.norm(b)
            basket[ci_val] = b / (norm + 1e-9)
    ci_arr = df_yr['ci'].values
    pi_arr = df_yr['pi'].values
    return (emb_pca[pi_arr] * basket[ci_arr]).sum(axis=1).astype(np.float32)

for yr, M_t, df_yr in [(2015, M_2015, df_2015), (2016, M_2016, df_2016)]:
    evaluate(yr, 'KNN (LLM embeddings)', knn_scores(M_t, df_yr))

Embeddings: (5018, 768)
PCA-LLM: (5018, 32)  (explains 61.6% variance)
[2015] KNN (LLM embeddings)        PR-AUC=0.0349  NDCG@20=0.0825  Prec@20=0.0761  CWR=0.6819  BestF1=0.0736  P@1000=0.1240  mAP@10=0.0353
[2016] KNN (LLM embeddings)        PR-AUC=0.0347  NDCG@20=0.0874  Prec@20=0.0783  CWR=0.6784  BestF1=0.0732  P@1000=0.1190  mAP@10=0.0414


## XGBoost — Full Universe Evaluation

Loads the pre-trained XGBoost checkpoint and scores every pair in the full M[t]=0 universe.
Features: 11 country (BACI+WDI) + 3 product + 32 PCA-LLM + density + ECI + 3×RCA-history = 51 dims.

In [39]:
import xgboost as xgb
import pickle as pkl
import os
import numpy as np
import pandas as pd

XGB_CKPT = os.path.join(DATA_DIR, 'models', 'xgboost', 'xgb_model.pkl')
if not os.path.exists(XGB_CKPT):
    raise FileNotFoundError(
        f'XGBoost checkpoint not found at {XGB_CKPT}. '
        'Run ClaudeFiles/train_xgboost.py first.')

with open(XGB_CKPT, 'rb') as _f:
    _xgb_bundle_fu = pkl.load(_f)
xgb_model_fu = _xgb_bundle_fu['model']
xgb_pca_fu   = _xgb_bundle_fu['pca']

# emb is already loaded by the KNN cell above
xgb_llm_pca_fu = xgb_pca_fu.transform(emb).astype('float32')
xgb_llm_pca_fu /= np.linalg.norm(xgb_llm_pca_fu, axis=1, keepdims=True).clip(min=1e-8)

_c_enrich_fu = pd.read_csv(os.path.join(DATA_DIR, 'country_features_enriched.csv'))
_p_feat_fu   = pd.read_csv(os.path.join(DATA_DIR, 'product_features.csv'))
_CCOLS_FU = ['log_export', 'n_products', 'avg_rca', 'max_rca',
             'gdp_pc', 'capital_formation', 'tertiary_enrollment',
             'fdi_inflows', 'manufacturing_va', 'internet_users', 'population']
_PCOLS_FU = ['log_world_export', 'ubiquity', 'avg_rca']
TRAIN_CUTOFF_FU = 2012

def _make_feat_arr(feat_df, id_col, idx_map, cols, n_ids):
    arr = np.zeros((n_ids, len(cols)), dtype=np.float32)
    avail = sorted(feat_df['year'].unique())
    yr = max(y for y in avail if y <= TRAIN_CUTOFF_FU)
    sub = feat_df[feat_df['year'] == yr].set_index(id_col)[cols]
    for eid, ei in idx_map.items():
        if eid in sub.index:
            arr[ei] = sub.loc[eid].values.astype(np.float32)
    return arr

_c_arr_fu = _make_feat_arr(_c_enrich_fu, 'country', c_idx, _CCOLS_FU, C)
_p_arr_fu = _make_feat_arr(_p_feat_fu,   'product', p_idx, _PCOLS_FU, P)
print(f'XGBoost loaded  |  PCA-LLM: {xgb_llm_pca_fu.shape}')

try:
    import cupy as cp
    _USE_CUPY_FU = True
    print('cupy available — GPU prediction enabled')
except ImportError:
    _USE_CUPY_FU = False
    print('cupy not available — using CPU prediction')

def build_xgb_features_fu(df_yr, t):
    """Fully vectorized 51-dim features. No Python loops."""
    ci = df_yr['ci'].values.astype(np.int64)
    pi = df_yr['pi'].values.astype(np.int64)

    # ECI and density
    M_t_fu = build_M(t)
    kc = M_t_fu.sum(axis=1); kp = M_t_fu.sum(axis=0)
    kcs = np.where(kc > 0, kc, 1.0); kps = np.where(kp > 0, kp, 1.0)
    kc_n, kp_n = kc.astype(float), kp.astype(float)
    for _ in range(20):
        kc_n = (1.0 / kcs) * (M_t_fu   @ kp_n)
        kp_n = (1.0 / kps) * (M_t_fu.T @ kc_n)
    eci_t  = (kc_n - kc_n.mean()) / (kc_n.std() + 1e-9)
    dens_t = (M_t_fu @ phi) / (phi_row_sum[None, :] + 1e-9)

    dens_batch = dens_t[ci, pi].reshape(-1, 1)
    eci_batch  = eci_t[ci].reshape(-1, 1)

    c_batch   = _c_arr_fu[ci]
    p_batch   = _p_arr_fu[pi]
    llm_batch = xgb_llm_pca_fu[pi]

    hist_yrs = [t - 2, t - 1, t]
    rca_sub  = rca_df[rca_df['year'].isin(hist_yrs)].copy()
    rca_sub['ci2'] = rca_sub['country'].map(c_idx)
    rca_sub['pi2'] = rca_sub['product'].map(p_idx)
    rca_sub  = rca_sub.dropna(subset=['ci2', 'pi2'])
    rca_sub['ci2'] = rca_sub['ci2'].astype(int)
    rca_sub['pi2'] = rca_sub['pi2'].astype(int)
    rca_arr  = np.zeros((C, P, 3), dtype=np.float32)
    for k, yr in enumerate(hist_yrs):
        rows = rca_sub[rca_sub['year'] == yr]
        rca_arr[rows['ci2'].values, rows['pi2'].values, k] = rows['rca'].values.astype(np.float32)
    rca_batch = rca_arr[ci, pi]

    return np.hstack([c_batch, p_batch, llm_batch, dens_batch, eci_batch, rca_batch])

def xgb_scores_universe(t, df_yr):
    """XGBoost probability scores over the full universe."""
    X = build_xgb_features_fu(df_yr, t).astype(np.float32)
    return xgb_model_fu.predict_proba(X)[:, 1].astype(np.float64)

for yr, df_yr in [(2015, df_2015), (2016, df_2016)]:
    print(f'  XGBoost full universe t={yr} ({len(df_yr):,} pairs)...')
    evaluate(yr, 'XGBoost', xgb_scores_universe(yr, df_yr))
print('XGBoost full-universe evaluation done.')

XGBoost loaded  |  PCA-LLM: (5018, 32)
cupy not available — using CPU prediction
  XGBoost full universe t=2015 (1,078,236 pairs)...
[2015] XGBoost                     PR-AUC=0.2276  NDCG@20=0.3585  Prec@20=0.3235  CWR=0.9575  BestF1=0.3130  P@1000=0.5190  mAP@10=0.2389
  XGBoost full universe t=2016 (1,079,598 pairs)...
[2016] XGBoost                     PR-AUC=0.2234  NDCG@20=0.3403  Prec@20=0.3095  CWR=0.9552  BestF1=0.3093  P@1000=0.4840  mAP@10=0.2150
XGBoost full-universe evaluation done.


## GNN Architecture & Checkpoint Loader

Loads pre-trained checkpoints from `data/models/gnn/checkpoints/`.
Generates scores for every pair in the full universe by running the GNN on the
5-year snapshot window ending at the observation year.

In [40]:
class _HomoGNN(nn.Module):
    def __init__(self, hidden, drop=0.3):
        super().__init__()
        self.c1 = SAGEConv(hidden, hidden)
        self.c2 = SAGEConv(hidden, hidden)
        self.drop = drop
    def forward(self, x, edge_index):
        x = F.dropout(self.c1(x, edge_index).relu(), p=self.drop, training=self.training)
        return self.c2(x, edge_index)

class BipartiteEncoder(nn.Module):
    def __init__(self, c_in, hidden, meta):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(3, hidden)
        self.gnn = to_hetero(_HomoGNN(hidden), meta)
    def forward(self, x_dict, ei_dict):
        return self.gnn({'country': self.country_lin(x_dict['country']),
                         'product': self.product_lin(x_dict['product'])}, ei_dict)

class TemporalGNN(nn.Module):
    def __init__(self, enc, hidden):
        super().__init__()
        self.enc   = enc
        self.gru_c = nn.GRU(hidden, hidden)
        self.gru_p = nn.GRU(hidden, hidden)
    def forward(self, snaps):
        cs, ps = [], []
        for s in snaps:
            z = self.enc(s.x_dict, s.edge_index_dict)
            cs.append(z['country']); ps.append(z['product'])
        z_c, _ = self.gru_c(torch.stack(cs))
        z_p, _ = self.gru_p(torch.stack(ps))
        return {'country': z_c[-1], 'product': z_p[-1]}

class LinkPredictor(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden, 1))
    def forward(self, zc, zp, ei):
        return self.mlp(torch.cat([zc[ei[0]], zp[ei[1]]], -1)).view(-1)

@torch.no_grad()
def gnn_scores_universe(ckpt_path, t, df_yr, c_x, with_cap=False):
    """
    Run GNN checkpoint on the full universe for observation year t.
    Returns score array aligned with df_yr rows.
    """
    ckpt = torch.load(ckpt_path, weights_only=False, map_location=DEVICE)
    enc  = BipartiteEncoder(ckpt['c_in'], ckpt['hidden'], ckpt['meta']).to(DEVICE)
    mdl  = TemporalGNN(enc, ckpt['hidden']).to(DEVICE)
    pred = LinkPredictor(ckpt['hidden']).to(DEVICE)
    mdl.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['mdl_state'].items()})
    pred.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['pred_state'].items()})
    mdl.eval(); pred.eval()

    # Build 5-year snapshot window
    snaps = []
    for y in range(t - 4, t + 1):
        d = HeteroData()
        d['country'].x = c_x[y].to(DEVICE)
        d['product'].x = p_x_by_yr[y].to(DEVICE)
        ei = edge_idx_by_yr[y].long().to(DEVICE)
        d['country', 'exports',     'product'].edge_index = ei
        d['product', 'rev_exports', 'country'].edge_index = ei.flip(0)
        if with_cap:
            d['product', 'capability', 'product'].edge_index = cap_ei.to(DEVICE)
        snaps.append(d)

    # Map country/product to GNN integer indices
    ci_gnn = df_yr['country'].map(c_map['to_idx'])
    pi_gnn = df_yr['product'].map(p_map['to_idx'])
    ok     = ci_gnn.notna() & pi_gnn.notna()
    ci_v   = ci_gnn[ok].astype(int).values
    pi_v   = pi_gnn[ok].astype(int).values
    ei_t   = torch.tensor([ci_v, pi_v], dtype=torch.long).to(DEVICE)

    z   = mdl(snaps)
    raw = torch.sigmoid(pred(z['country'], z['product'], ei_t)).cpu().numpy()

    # Align back to df_yr (unmapped pairs get score=0)
    score_series = pd.Series(0.0, index=df_yr.index)
    score_series[ok[ok].index] = raw
    n_unmapped = (~ok).sum()
    if n_unmapped > 0:
        print(f'  Warning: {n_unmapped} pairs had no GNN mapping (scored 0)')
    return score_series.values

print('GNN helpers loaded.')

GNN helpers loaded.


## GNN-LLM v2 Architecture & Checkpoint Loader

v2 upgrades over the original GNN-11F+LLM:

| Component | v1 (SAGEConv) | v2 (GATConv) |
|-----------|--------------|--------------|
| GNN layer | SAGEConv — unweighted mean | GATConv — learned attention, configurable heads |
| Product features | 3 BACI features | 3 BACI + 768 LLM = **771 features** |
| Capability edge weights | None (binary topology) | **Cosine similarity** as `edge_attr` |
| Loss | BCEWithLogitsLoss + pos_weight | **Binary Focal Loss** (α, γ from Optuna) |
| Hyperparameters | Fixed | **Optuna** (40 trials, TPE sampler) |

Checkpoint: `data/models/gnn/checkpoints/gnn_llm_v2.pt`

In [41]:
class _GATBlock(nn.Module):
    def __init__(self, hidden, heads, drop):
        super().__init__()
        self.gat1 = GATConv(hidden, hidden, heads=heads, concat=True,
                             dropout=drop, edge_dim=1,
                             add_self_loops=False, fill_value='mean')
        self.proj = nn.Linear(hidden * heads, hidden)
        self.gat2 = GATConv(hidden, hidden, heads=1, concat=False,
                             dropout=drop, edge_dim=1,
                             add_self_loops=False, fill_value='mean')
        self.drop = drop

    def forward(self, x, edge_index, edge_attr=None):
        x = F.dropout(self.proj(self.gat1(x, edge_index, edge_attr=edge_attr).relu()),
                      p=self.drop, training=self.training)
        return self.gat2(x, edge_index, edge_attr=edge_attr)


class BipartiteEncoderGAT(nn.Module):
    def __init__(self, c_in, p_in, hidden, heads, drop, meta):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(p_in, hidden)
        self.gnn = to_hetero(_GATBlock(hidden, heads, drop), meta)

    def forward(self, x_dict, ei_dict, ea_dict=None):
        x_proj = {
            'country': self.country_lin(x_dict['country']),
            'product': self.product_lin(x_dict['product']),
        }
        if ea_dict is not None:
            return self.gnn(x_proj, ei_dict, ea_dict)
        return self.gnn(x_proj, ei_dict)


class TemporalGNNv2(nn.Module):
    def __init__(self, enc, hidden):
        super().__init__()
        self.enc   = enc
        self.gru_c = nn.GRU(hidden, hidden, batch_first=False)
        self.gru_p = nn.GRU(hidden, hidden, batch_first=False)

    def forward(self, snaps):
        cs, ps = [], []
        for s in snaps:
            ea = getattr(s, '_ea_dict', None)
            z  = self.enc(s.x_dict, s.edge_index_dict, ea)
            cs.append(z['country']); ps.append(z['product'])
        z_c, _ = self.gru_c(torch.stack(cs))
        z_p, _ = self.gru_p(torch.stack(ps))
        return {'country': z_c[-1], 'product': z_p[-1]}


@torch.no_grad()
def gnn_scores_universe_v2(ckpt_path, t, df_yr):
    """
    Run v2 checkpoint (GATConv + LLM features + cosine edge weights)
    on the full universe for observation year t.
    """
    ckpt = torch.load(ckpt_path, weights_only=False, map_location=DEVICE)
    enc  = BipartiteEncoderGAT(
        ckpt['c_in'], ckpt['p_in'], ckpt['hidden'], ckpt['heads'], ckpt['dropout'],
        ckpt['meta']
    ).to(DEVICE)
    mdl  = TemporalGNNv2(enc, ckpt['hidden']).to(DEVICE)
    pred = LinkPredictor(ckpt['hidden']).to(DEVICE)
    mdl.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['mdl_state'].items()})
    pred.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['pred_state'].items()})
    mdl.eval(); pred.eval()

    cap_ei_dev = cap_ei.to(DEVICE)
    cos_w_dev  = cos_weights.to(DEVICE)

    snaps = []
    for y in range(t - 4, t + 1):
        d = HeteroData()
        d['country'].x = c_x_11feat[y].to(DEVICE)
        d['product'].x = p_x_by_yr_v2[y].to(DEVICE)      # [5018, 771]
        ei = edge_idx_by_yr[y].long().to(DEVICE)
        d['country', 'exports',     'product'].edge_index = ei
        d['product', 'rev_exports', 'country'].edge_index = ei.flip(0)
        d['product', 'capability',  'product'].edge_index = cap_ei_dev
        d['product', 'capability',  'product'].edge_attr  = cos_w_dev
        d._ea_dict = {
            ('country', 'exports',     'product'): None,
            ('product', 'rev_exports', 'country'): None,
            ('product', 'capability',  'product'): cos_w_dev,
        }
        snaps.append(d)

    ci_gnn = df_yr['country'].map(c_map['to_idx'])
    pi_gnn = df_yr['product'].map(p_map['to_idx'])
    ok     = ci_gnn.notna() & pi_gnn.notna()
    ci_v   = ci_gnn[ok].astype(int).values
    pi_v   = pi_gnn[ok].astype(int).values
    ei_t   = torch.tensor([ci_v, pi_v], dtype=torch.long).to(DEVICE)

    z   = mdl(snaps)
    raw = torch.sigmoid(pred(z['country'], z['product'], ei_t)).cpu().numpy()

    score_series = pd.Series(0.0, index=df_yr.index)
    score_series[ok[ok].index] = raw
    n_unmapped = (~ok).sum()
    if n_unmapped > 0:
        print(f'  Warning: {n_unmapped} pairs had no GNN mapping (scored 0)')
    return score_series.values


# Try both possible checkpoint filenames (training notebook saves to gnn_llm_v2.pt,
# but the file on disk may also be named gnn_11f_llm_v2.pt)
_v2_candidates = ['gnn_llm_v2.pt', 'gnn_11f_llm_v2.pt']
CKPT_V2 = None
for _name in _v2_candidates:
    _path = os.path.join(CKPT_DIR, _name)
    if os.path.exists(_path):
        CKPT_V2 = _path
        break

v2_exists = CKPT_V2 is not None
print(f'v2 checkpoint search: {_v2_candidates}')
if v2_exists:
    ck = torch.load(CKPT_V2, weights_only=False, map_location='cpu')
    print(f'  Found: {CKPT_V2}')
    print(f'  val PR-AUC={ck.get("best_val_prauc", "?"):.4f}  '
          f'hidden={ck.get("hidden")}  heads={ck.get("heads")}  p_in={ck.get("p_in")}')
else:
    print('  NOT FOUND — run new_gnn_training.ipynb first.')

v2 checkpoint search: ['gnn_llm_v2.pt', 'gnn_11f_llm_v2.pt']
  Found: data\models\gnn\checkpoints\gnn_11f_llm_v2.pt
  val PR-AUC=0.4189  hidden=64  heads=2  p_in=771


In [42]:
# ── Variant A: SAGEConv + PCA features ─────────────────────────────────────
class _HomoGNN_SAGE(nn.Module):
    def __init__(self, hidden, drop=0.3):
        super().__init__()
        self.c1   = SAGEConv(hidden, hidden)
        self.c2   = SAGEConv(hidden, hidden)
        self.drop = drop
    def forward(self, x, edge_index):
        x = F.dropout(self.c1(x, edge_index).relu(), p=self.drop, training=self.training)
        return self.c2(x, edge_index)

class BipartiteEncoderSAGE_PCA(nn.Module):
    def __init__(self, c_in, p_in, hidden, drop, meta):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(p_in, hidden)
        self.gnn = to_hetero(_HomoGNN_SAGE(hidden, drop), meta)
    def forward(self, snap, _ew=None):
        x_proj = {'country': self.country_lin(snap['country'].x),
                  'product': self.product_lin(snap['product'].x)}
        return self.gnn(x_proj, snap.edge_index_dict)

class TemporalGNN_PCA(nn.Module):
    # Accepts list of (HeteroData, ew|None) tuples
    def __init__(self, enc, hidden):
        super().__init__()
        self.enc   = enc
        self.gru_c = nn.GRU(hidden, hidden)
        self.gru_p = nn.GRU(hidden, hidden)
    def forward(self, snaps):
        cs, ps = [], []
        for s, ew in snaps:
            z = self.enc(s, ew)
            cs.append(z['country']); ps.append(z['product'])
        z_c, _ = self.gru_c(torch.stack(cs))
        z_p, _ = self.gru_p(torch.stack(ps))
        return {'country': z_c[-1], 'product': z_p[-1]}

# ── Variant B: Mixed SAGEConv + GCNConv with cosine edge weights ─────────────
class MixedBipartiteEncoder(nn.Module):
    def __init__(self, c_in, p_in, hidden, drop):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(p_in, hidden)
        self.drop = drop
        self.sage_exp_1  = SAGEConv(hidden, hidden)
        self.sage_rexp_1 = SAGEConv(hidden, hidden)
        self.gcn_cap_1   = GCNConv(hidden, hidden, add_self_loops=False, normalize=False)
        self.sage_exp_2  = SAGEConv(hidden, hidden)
        self.sage_rexp_2 = SAGEConv(hidden, hidden)
        self.gcn_cap_2   = GCNConv(hidden, hidden, add_self_loops=False, normalize=False)

    def _layer(self, xc, xp, snap, cap_ew, sage_exp, sage_rexp, gcn_cap):
        ei_exp  = snap['country', 'exports',     'product'].edge_index
        ei_rexp = snap['product', 'rev_exports', 'country'].edge_index
        ei_cap  = snap['product', 'capability',  'product'].edge_index
        xc_new  = sage_rexp((xp, xc), ei_rexp)
        xp_new  = sage_exp((xc, xp), ei_exp) + gcn_cap(xp, ei_cap, edge_weight=cap_ew)
        return xc_new, xp_new

    def forward(self, snap, cap_ew):
        xc = self.country_lin(snap['country'].x)
        xp = self.product_lin(snap['product'].x)
        xc, xp = self._layer(xc, xp, snap, cap_ew,
                              self.sage_exp_1, self.sage_rexp_1, self.gcn_cap_1)
        xc = F.dropout(xc.relu(), p=self.drop, training=self.training)
        xp = F.dropout(xp.relu(), p=self.drop, training=self.training)
        xc, xp = self._layer(xc, xp, snap, cap_ew,
                              self.sage_exp_2, self.sage_rexp_2, self.gcn_cap_2)
        return {'country': xc, 'product': xp}

# ── Full-universe scorer for PCA-based models ─────────────────────────────────
@torch.no_grad()
def gnn_scores_universe_pca(ckpt_path, t, df_yr, variant='A'):
    # variant='A': BipartiteEncoderSAGE_PCA + TemporalGNN_PCA (no edge weights)
    # variant='B': MixedBipartiteEncoder + TemporalGNN_PCA (cosine weights via GCNConv)
    ckpt   = torch.load(ckpt_path, weights_only=False, map_location=DEVICE)
    hidden = ckpt.get('hidden', 128)
    p_in   = ckpt.get('p_in', P_IN_PCA)

    cap_ei_dev = cap_ei.to(DEVICE)
    ew_dev     = cos_weights_1d.to(DEVICE)

    snaps = []
    for y in range(t - 4, t + 1):
        d = HeteroData()
        d['country'].x = c_x_11feat[y].to(DEVICE)
        d['product'].x = p_x_with_pca[y].to(DEVICE)
        ei = edge_idx_by_yr[y].long().to(DEVICE)
        d['country', 'exports',     'product'].edge_index = ei
        d['product', 'rev_exports', 'country'].edge_index = ei.flip(0)
        d['product', 'capability',  'product'].edge_index = cap_ei_dev
        snaps.append((d, ew_dev if variant == 'B' else None))

    meta = snaps[0][0].metadata()
    if variant == 'A':
        enc = BipartiteEncoderSAGE_PCA(11, p_in, hidden, drop=0.3, meta=meta).to(DEVICE)
    else:
        enc = MixedBipartiteEncoder(11, p_in, hidden, drop=0.3).to(DEVICE)

    mdl  = TemporalGNN_PCA(enc, hidden).to(DEVICE)
    pred = LinkPredictor(hidden).to(DEVICE)
    mdl.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['mdl_state'].items()})
    pred.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['pred_state'].items()})
    mdl.eval(); pred.eval()

    ci_gnn = df_yr['country'].map(c_map['to_idx'])
    pi_gnn = df_yr['product'].map(p_map['to_idx'])
    ok     = ci_gnn.notna() & pi_gnn.notna()
    ci_v   = ci_gnn[ok].astype(int).values
    pi_v   = pi_gnn[ok].astype(int).values
    ei_t   = torch.tensor([ci_v, pi_v], dtype=torch.long).to(DEVICE)

    z   = mdl(snaps)
    raw = torch.sigmoid(pred(z['country'], z['product'], ei_t)).cpu().numpy()

    score_series = pd.Series(0.0, index=df_yr.index)
    score_series[ok[ok].index] = raw
    n_unmapped = (~ok).sum()
    if n_unmapped > 0:
        print(f'  Warning: {n_unmapped} pairs had no GNN mapping (scored 0)')
    return score_series.values


CKPT_PCA_A = os.path.join(CKPT_DIR, 'gnn_11f_llm_pca.pt')
CKPT_PCA_B = os.path.join(CKPT_DIR, 'gnn_11f_llm_pca_ew.pt')

for name, path_ck in [('PCA-A (SAGE+PCA)', CKPT_PCA_A), ('PCA-B (GCN+EW)', CKPT_PCA_B)]:
    status = 'OK' if os.path.exists(path_ck) else 'NOT FOUND — run new_gnn_training_fixed.ipynb first'
    print(f'  {name}: {status}')
print('PCA architecture + scorers ready.')


  PCA-A (SAGE+PCA): OK
  PCA-B (GCN+EW): OK
PCA architecture + scorers ready.


In [43]:
# ── Variants C & D: GATConv + PCA features (Optuna-optimized) ───────────────
class _GATBlock_PCA(nn.Module):
    def __init__(self, hidden, heads, drop):
        super().__init__()
        self.gat1 = GATConv(hidden, hidden, heads=heads, concat=True,
                             dropout=drop, edge_dim=1, add_self_loops=False, fill_value='mean')
        self.proj = nn.Linear(hidden * heads, hidden)
        self.gat2 = GATConv(hidden, hidden, heads=1, concat=False,
                             dropout=drop, edge_dim=1, add_self_loops=False, fill_value='mean')
        self.drop = drop
    def forward(self, x, edge_index, edge_attr=None):
        x = F.dropout(self.proj(self.gat1(x, edge_index, edge_attr=edge_attr).relu()),
                      p=self.drop, training=self.training)
        return self.gat2(x, edge_index, edge_attr=edge_attr)

class BipartiteEncoderGAT_PCA(nn.Module):
    def __init__(self, c_in, p_in, hidden, heads, drop, meta):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(p_in, hidden)
        self.gnn = to_hetero(_GATBlock_PCA(hidden, heads, drop), meta)
    def forward(self, x_dict, ei_dict, ea_dict=None):
        x_proj = {'country': self.country_lin(x_dict['country']),
                  'product': self.product_lin(x_dict['product'])}
        return self.gnn(x_proj, ei_dict, ea_dict) if ea_dict else self.gnn(x_proj, ei_dict)

class TemporalGNN_GAT_PCA(nn.Module):
    def __init__(self, enc, hidden):
        super().__init__()
        self.enc   = enc
        self.gru_c = nn.GRU(hidden, hidden)
        self.gru_p = nn.GRU(hidden, hidden)
    def forward(self, snaps):
        cs, ps = [], []
        for s, ea in snaps:
            z = self.enc(s.x_dict, s.edge_index_dict, ea)
            cs.append(z['country']); ps.append(z['product'])
        z_c, _ = self.gru_c(torch.stack(cs))
        z_p, _ = self.gru_p(torch.stack(ps))
        return {'country': z_c[-1], 'product': z_p[-1]}


@torch.no_grad()
def gnn_scores_universe_gat_pca(ckpt_path, t, df_yr, variant='C'):
    ckpt   = torch.load(ckpt_path, weights_only=False, map_location=DEVICE)
    hidden = ckpt.get('hidden', 128)
    heads  = ckpt.get('heads', 2)
    drop   = ckpt.get('dropout', 0.3)
    p_in   = ckpt.get('p_in', P_IN_PCA)

    cap_ei_dev = cap_ei.to(DEVICE)
    ew_dev     = cos_weights_1d.unsqueeze(1).to(DEVICE)   # [E, 1] for GATConv

    snaps = []
    for y in range(t - 4, t + 1):
        d = HeteroData()
        d['country'].x = c_x_11feat[y].to(DEVICE)
        d['product'].x = p_x_with_pca[y].to(DEVICE)
        ei = edge_idx_by_yr[y].long().to(DEVICE)
        d['country', 'exports',     'product'].edge_index = ei
        d['product', 'rev_exports', 'country'].edge_index = ei.flip(0)
        d['product', 'capability',  'product'].edge_index = cap_ei_dev
        if variant == 'D':
            d['product', 'capability', 'product'].edge_attr = ew_dev
            ea = {
                ('country', 'exports',     'product'): None,
                ('product', 'rev_exports', 'country'): None,
                ('product', 'capability',  'product'): ew_dev,
            }
        else:
            ea = None
        snaps.append((d, ea))

    meta = snaps[0][0].metadata()
    enc  = BipartiteEncoderGAT_PCA(11, p_in, hidden, heads, drop, meta).to(DEVICE)
    mdl  = TemporalGNN_GAT_PCA(enc, hidden).to(DEVICE)
    pred = LinkPredictor(hidden).to(DEVICE)
    mdl.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['mdl_state'].items()})
    pred.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['pred_state'].items()})
    mdl.eval(); pred.eval()

    ci_gnn = df_yr['country'].map(c_map['to_idx'])
    pi_gnn = df_yr['product'].map(p_map['to_idx'])
    ok     = ci_gnn.notna() & pi_gnn.notna()
    ci_v   = ci_gnn[ok].astype(int).values
    pi_v   = pi_gnn[ok].astype(int).values
    ei_t   = torch.tensor([ci_v, pi_v], dtype=torch.long).to(DEVICE)

    z   = mdl(snaps)
    raw = torch.sigmoid(pred(z['country'], z['product'], ei_t)).cpu().numpy()

    score_series = pd.Series(0.0, index=df_yr.index)
    score_series[ok[ok].index] = raw
    n_unmapped = (~ok).sum()
    if n_unmapped > 0:
        print(f'  Warning: {n_unmapped} pairs had no GNN mapping (scored 0)')
    return score_series.values


CKPT_PCA_C = os.path.join(CKPT_DIR, 'gnn_11f_llm_pca_gat.pt')
CKPT_PCA_D = os.path.join(CKPT_DIR, 'gnn_11f_llm_pca_gat_ew.pt')

for name, path_ck in [('PCA-C (GAT opt)', CKPT_PCA_C), ('PCA-D (GAT+EW opt)', CKPT_PCA_D)]:
    status = 'OK' if os.path.exists(path_ck) else 'NOT FOUND — run new_gnn_training_fixed.ipynb first'
    print(f'  {name}: {status}')
print('GAT-PCA architecture + scorers ready.')


  PCA-C (GAT opt): NOT FOUND — run new_gnn_training_fixed.ipynb first
  PCA-D (GAT+EW opt): OK
GAT-PCA architecture + scorers ready.


## Method 6 — GNN-4F (BACI only)

In [44]:
CKPT_4F = os.path.join(CKPT_DIR, 'gnn_4f.pt')
for yr, df_yr in [(2015, df_2015), (2016, df_2016)]:
    print(f'  Loading GNN-4F for t={yr}...')
    evaluate(yr, 'GNN-4F', gnn_scores_universe(CKPT_4F, yr, df_yr, c_x_4feat, with_cap=False))

  Loading GNN-4F for t=2015...
[2015] GNN-4F                      PR-AUC=0.0727  NDCG@20=0.1048  Prec@20=0.1011  CWR=0.9141  BestF1=0.1356  P@1000=0.1620  mAP@10=0.0391
  Loading GNN-4F for t=2016...
[2016] GNN-4F                      PR-AUC=0.0764  NDCG@20=0.1036  Prec@20=0.0954  CWR=0.9106  BestF1=0.1417  P@1000=0.1800  mAP@10=0.0432


## Method 7 — GNN-11F (BACI + WDI)

In [45]:
CKPT_11F = os.path.join(CKPT_DIR, 'gnn_11f.pt')
for yr, df_yr in [(2015, df_2015), (2016, df_2016)]:
    print(f'  Loading GNN-11F for t={yr}...')
    evaluate(yr, 'GNN-11F (BACI+WDI)', gnn_scores_universe(CKPT_11F, yr, df_yr, c_x_11feat, with_cap=False))

  Loading GNN-11F for t=2015...
[2015] GNN-11F (BACI+WDI)          PR-AUC=0.0811  NDCG@20=0.1384  Prec@20=0.1246  CWR=0.9241  BestF1=0.1472  P@1000=0.1850  mAP@10=0.0681
  Loading GNN-11F for t=2016...
[2016] GNN-11F (BACI+WDI)          PR-AUC=0.0816  NDCG@20=0.1301  Prec@20=0.1144  CWR=0.9182  BestF1=0.1477  P@1000=0.2050  mAP@10=0.0674


## Method 8 — GNN-11F+LLM

In [46]:
CKPT_LLM = os.path.join(CKPT_DIR, 'gnn_11f_llm.pt')
for yr, df_yr in [(2015, df_2015), (2016, df_2016)]:
    print(f'  Loading GNN-11F+LLM for t={yr}...')
    evaluate(yr, 'GNN-11F+LLM', gnn_scores_universe(CKPT_LLM, yr, df_yr, c_x_11feat, with_cap=True))

  Loading GNN-11F+LLM for t=2015...
[2015] GNN-11F+LLM                 PR-AUC=0.0846  NDCG@20=0.1605  Prec@20=0.1409  CWR=0.9258  BestF1=0.1521  P@1000=0.2430  mAP@10=0.0859
  Loading GNN-11F+LLM for t=2016...
[2016] GNN-11F+LLM                 PR-AUC=0.0837  NDCG@20=0.1596  Prec@20=0.1405  CWR=0.9172  BestF1=0.1501  P@1000=0.2420  mAP@10=0.0894


## Method 9 — GNN-LLM v2 (GAT + Focal Loss + Optuna)

Revamped model: GATConv with learned attention, 768-dim LLM embeddings as product features,
cosine-similarity edge weights on capability edges, Binary Focal Loss, Optuna-tuned hyperparameters.

Skipped gracefully if checkpoint `gnn_llm_v2.pt` does not yet exist.

In [47]:
if v2_exists:
    for yr, df_yr in [(2015, df_2015), (2016, df_2016)]:
        print(f'  Loading GNN-LLM v2 for t={yr}...')
        evaluate(yr, 'GNN-LLM v2 (GAT+Focal)', gnn_scores_universe_v2(CKPT_V2, yr, df_yr))
else:
    print('Skipping GNN-LLM v2 — checkpoint not found.')
    print('Run new_gnn_training.ipynb to train and save gnn_llm_v2.pt')

  Loading GNN-LLM v2 for t=2015...
[2015] GNN-LLM v2 (GAT+Focal)      PR-AUC=0.0741  NDCG@20=0.1326  Prec@20=0.1206  CWR=0.9210  BestF1=0.1274  P@1000=0.2540  mAP@10=0.0692
  Loading GNN-LLM v2 for t=2016...
[2016] GNN-LLM v2 (GAT+Focal)      PR-AUC=0.0730  NDCG@20=0.1309  Prec@20=0.1155  CWR=0.9133  BestF1=0.1280  P@1000=0.2340  mAP@10=0.0718


## Method 10 — GNN-LLM v2 Unopt (GAT + Focal Loss, fixed hparams)

Same architecture as Method 9 (GATConv + 771-dim product features + focal loss) but trained with
**fixed defaults** instead of Optuna-tuned hyperparameters: hidden=128, heads=4, dropout=0.3,
lr=1e-3, focal_alpha=0.5, focal_gamma=2.0.

Checkpoint: `data/models/gnn/checkpoints/gnn_llm_v2_unopt.pt`  
Val PR-AUC: 0.3211 (vs. 0.4189 for the Optuna variant)

In [48]:
@torch.no_grad()
def gnn_scores_universe_v2_fixed(ckpt_path, t, df_yr):
    """
    Run v2 checkpoint using tuple-based ea_dict (avoids PyG _dict interception).
    Compatible with both 'gnn_11f_llm_v2.pt' and 'gnn_llm_v2_unopt.pt'.
    """
    ckpt = torch.load(ckpt_path, weights_only=False, map_location=DEVICE)
    enc  = BipartiteEncoderGAT(
        ckpt['c_in'], ckpt['p_in'], ckpt['hidden'], ckpt['heads'], ckpt['dropout'],
        ckpt['meta']
    ).to(DEVICE)

    # TemporalGNNv2 that accepts list of (HeteroData, ea_dict) tuples
    class _TemporalGNNv2_Fixed(nn.Module):
        def __init__(self, enc, hidden):
            super().__init__()
            self.enc   = enc
            self.gru_c = nn.GRU(hidden, hidden, batch_first=False)
            self.gru_p = nn.GRU(hidden, hidden, batch_first=False)
        def forward(self, snaps):
            cs, ps = [], []
            for s, ea in snaps:
                z = self.enc(s.x_dict, s.edge_index_dict, ea)
                cs.append(z['country']); ps.append(z['product'])
            z_c, _ = self.gru_c(torch.stack(cs))
            z_p, _ = self.gru_p(torch.stack(ps))
            return {'country': z_c[-1], 'product': z_p[-1]}

    mdl  = _TemporalGNNv2_Fixed(enc, ckpt['hidden']).to(DEVICE)
    pred = LinkPredictor(ckpt['hidden']).to(DEVICE)
    mdl.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['mdl_state'].items()})
    pred.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['pred_state'].items()})
    mdl.eval(); pred.eval()

    cap_ei_dev = cap_ei.to(DEVICE)
    cos_w_dev  = cos_weights.to(DEVICE)

    snaps = []
    for y in range(t - 4, t + 1):
        d = HeteroData()
        d['country'].x = c_x_11feat[y].to(DEVICE)
        d['product'].x = p_x_by_yr_v2[y].to(DEVICE)
        ei = edge_idx_by_yr[y].long().to(DEVICE)
        d['country', 'exports',     'product'].edge_index = ei
        d['product', 'rev_exports', 'country'].edge_index = ei.flip(0)
        d['product', 'capability',  'product'].edge_index = cap_ei_dev
        d['product', 'capability',  'product'].edge_attr  = cos_w_dev
        ea = {
            ('country', 'exports',     'product'): None,
            ('product', 'rev_exports', 'country'): None,
            ('product', 'capability',  'product'): cos_w_dev,
        }
        snaps.append((d, ea))

    ci_gnn = df_yr['country'].map(c_map['to_idx'])
    pi_gnn = df_yr['product'].map(p_map['to_idx'])
    ok     = ci_gnn.notna() & pi_gnn.notna()
    ci_v   = ci_gnn[ok].astype(int).values
    pi_v   = pi_gnn[ok].astype(int).values
    ei_t   = torch.tensor([ci_v, pi_v], dtype=torch.long).to(DEVICE)

    z   = mdl(snaps)
    raw = torch.sigmoid(pred(z['country'], z['product'], ei_t)).cpu().numpy()

    score_series = pd.Series(0.0, index=df_yr.index)
    score_series[ok[ok].index] = raw
    n_unmapped = (~ok).sum()
    if n_unmapped > 0:
        print(f'  Warning: {n_unmapped} pairs had no GNN mapping (scored 0)')
    return score_series.values


CKPT_V2_UNOPT = os.path.join(CKPT_DIR, 'gnn_llm_v2_unopt.pt')

if os.path.exists(CKPT_V2_UNOPT):
    ck = torch.load(CKPT_V2_UNOPT, weights_only=False, map_location='cpu')
    print(f'v2 Unopt checkpoint: {CKPT_V2_UNOPT}')
    print(f'  val PR-AUC={ck.get("best_val_prauc", "?"):.4f}  '
          f'hidden={ck.get("hidden")}  heads={ck.get("heads")}  p_in={ck.get("p_in")}  '
          f'optimized={ck.get("optimized")}')
    for yr, df_yr in [(2015, df_2015), (2016, df_2016)]:
        print(f'  Loading GNN-LLM v2 Unopt for t={yr}...')
        evaluate(yr, 'GNN-LLM v2 Unopt', gnn_scores_universe_v2_fixed(CKPT_V2_UNOPT, yr, df_yr))
else:
    print(f'CKPT NOT FOUND: {CKPT_V2_UNOPT}')

v2 Unopt checkpoint: data\models\gnn\checkpoints\gnn_llm_v2_unopt.pt
  val PR-AUC=0.3211  hidden=128  heads=4  p_in=771  optimized=False
  Loading GNN-LLM v2 Unopt for t=2015...
[2015] GNN-LLM v2 Unopt            PR-AUC=0.0473  NDCG@20=0.0813  Prec@20=0.0699  CWR=0.8845  BestF1=0.0940  P@1000=0.0670  mAP@10=0.0444
  Loading GNN-LLM v2 Unopt for t=2016...
[2016] GNN-LLM v2 Unopt            PR-AUC=0.0491  NDCG@20=0.0741  Prec@20=0.0666  CWR=0.8812  BestF1=0.0987  P@1000=0.0660  mAP@10=0.0360


In [49]:
# ── Method 11 & 12: GNN-LLM PCA-A and PCA-B ─────────────────────────────────
for yr, df_yr in [(2015, df_2015), (2016, df_2016)]:
    if os.path.exists(CKPT_PCA_A):
        print(f'  Loading GNN-LLM PCA-A (SAGEConv+PCA) for t={yr}...')
        evaluate(yr, 'GNN-LLM PCA-A (SAGE)', gnn_scores_universe_pca(CKPT_PCA_A, yr, df_yr, variant='A'))
    else:
        print(f'  GNN-LLM PCA-A NOT FOUND — run new_gnn_training_fixed.ipynb first')
        break

for yr, df_yr in [(2015, df_2015), (2016, df_2016)]:
    if os.path.exists(CKPT_PCA_B):
        print(f'  Loading GNN-LLM PCA-B (GCNConv+EW) for t={yr}...')
        evaluate(yr, 'GNN-LLM PCA-B (GCN+EW)', gnn_scores_universe_pca(CKPT_PCA_B, yr, df_yr, variant='B'))
    else:
        print(f'  GNN-LLM PCA-B NOT FOUND — run new_gnn_training_fixed.ipynb first')
        break


  Loading GNN-LLM PCA-A (SAGEConv+PCA) for t=2015...
[2015] GNN-LLM PCA-A (SAGE)        PR-AUC=0.0877  NDCG@20=0.1520  Prec@20=0.1400  CWR=0.9289  BestF1=0.1529  P@1000=0.2370  mAP@10=0.0774
  Loading GNN-LLM PCA-A (SAGEConv+PCA) for t=2016...
[2016] GNN-LLM PCA-A (SAGE)        PR-AUC=0.0906  NDCG@20=0.1510  Prec@20=0.1376  CWR=0.9273  BestF1=0.1570  P@1000=0.2770  mAP@10=0.0766
  Loading GNN-LLM PCA-B (GCNConv+EW) for t=2015...
[2015] GNN-LLM PCA-B (GCN+EW)      PR-AUC=0.0914  NDCG@20=0.1659  Prec@20=0.1438  CWR=0.9330  BestF1=0.1558  P@1000=0.2630  mAP@10=0.0904
  Loading GNN-LLM PCA-B (GCNConv+EW) for t=2016...
[2016] GNN-LLM PCA-B (GCN+EW)      PR-AUC=0.0897  NDCG@20=0.1597  Prec@20=0.1367  CWR=0.9264  BestF1=0.1550  P@1000=0.2670  mAP@10=0.0854


In [50]:
# ── Methods 13 & 14: GNN-LLM PCA-C and PCA-D (GAT + Focal + Optuna) ─────────
for yr, df_yr in [(2015, df_2015), (2016, df_2016)]:
    if os.path.exists(CKPT_PCA_C):
        print(f'  Loading GNN-LLM PCA-C (GAT+PCA, Optuna) for t={yr}...')
        evaluate(yr, 'GNN-LLM PCA-C (GAT opt)', gnn_scores_universe_gat_pca(CKPT_PCA_C, yr, df_yr, variant='C'))
    else:
        print(f'  GNN-LLM PCA-C NOT FOUND — run new_gnn_training_fixed.ipynb first')
        break

for yr, df_yr in [(2015, df_2015), (2016, df_2016)]:
    if os.path.exists(CKPT_PCA_D):
        print(f'  Loading GNN-LLM PCA-D (GAT+EW, Optuna) for t={yr}...')
        evaluate(yr, 'GNN-LLM PCA-D (GAT+EW opt)', gnn_scores_universe_gat_pca(CKPT_PCA_D, yr, df_yr, variant='D'))
    else:
        print(f'  GNN-LLM PCA-D NOT FOUND — run new_gnn_training_fixed.ipynb first')
        break


  GNN-LLM PCA-C NOT FOUND — run new_gnn_training_fixed.ipynb first
  Loading GNN-LLM PCA-D (GAT+EW, Optuna) for t=2015...
[2015] GNN-LLM PCA-D (GAT+EW opt)  PR-AUC=0.0746  NDCG@20=0.1407  Prec@20=0.1166  CWR=0.9239  BestF1=0.1271  P@1000=0.2480  mAP@10=0.0773
  Loading GNN-LLM PCA-D (GAT+EW, Optuna) for t=2016...
[2016] GNN-LLM PCA-D (GAT+EW opt)  PR-AUC=0.0725  NDCG@20=0.1310  Prec@20=0.1069  CWR=0.9178  BestF1=0.1278  P@1000=0.2290  mAP@10=0.0734


## Results Tables — Full Universe

Saves CSVs to `full_universe_eval/` and prints formatted comparison tables.

In [58]:
METHOD_ORDER = [
    'RCA Persistence', 'Density', 'ECI', 'ECI + Density',
    'KNN (LLM embeddings)', 'XGBoost', 'GNN-4F', 'GNN-11F (BACI+WDI)',
    'GNN-11F+LLM', 'GNN-LLM v2 (GAT+Focal)', 'GNN-LLM v2 Unopt',
    'GNN-LLM PCA-A (SAGE)', 'GNN-LLM PCA-B (GCN+EW)',
    'GNN-LLM PCA-C (GAT opt)', 'GNN-LLM PCA-D (GAT+EW opt)',
]
METRICS = ['PR-AUC', 'AUROC', 'NDCG@20', 'Prec@20', 'CWR', 'Best F1', 'P@1000', 'mAP@10']

for yr in [2015, 2016]:
    df_yr     = df_2015 if yr == 2015 else df_2016
    n_pairs   = len(df_yr)
    pos_rate  = df_yr['label'].mean() * 100
    methods   = [m for m in METHOD_ORDER if m in RESULTS[yr]]

    print(f'\n{"="*102}')
    print(f'  FULL-UNIVERSE EVALUATION — t={yr}  ({n_pairs:,} pairs, {pos_rate:.2f}% positive)')
    print(f'{"="*102}')
    print(f'  {"Method":<30}' + ''.join(f'{m:>9}' for m in METRICS))
    print(f'  {"-"*102}')

    rows = []
    for m in methods:
        r   = RESULTS[yr][m]
        tag = ' <--' if 'GNN' in m else ''
        row_dict = {'Method': m}
        vals = []
        for met in METRICS:
            v = r.get(met, 'N/A')
            row_dict[met] = v
            vals.append(f'{"N/A":>9}' if v == 'N/A' else f'{v:>9.4f}')
        print(f'  {m:<30}' + ''.join(vals) + tag)
        rows.append(row_dict)

    print(f'{"="*102}')
    print('  N/A = ECI has no product-level ranking signal.')
    print('  CWR uses percentile-ranked scores (top 50% = predicted positive).')
    print('  CWR weights: 1/ubiquity[2010]; missing products use median ubiquity (bug-fixed).')

    df_res = pd.DataFrame(rows).set_index('Method')
    csv_path = os.path.join(OUT_DIR, f'full_universe_{yr}_results.csv')
    df_res.to_csv(csv_path)
    print(f'\n  Saved -> {csv_path}')

print('\nAll results saved.')


  FULL-UNIVERSE EVALUATION — t=2015  (1,078,236 pairs, 1.71% positive)
  Method                           PR-AUC    AUROC  NDCG@20  Prec@20      CWR  Best F1   P@1000   mAP@10
  ------------------------------------------------------------------------------------------------------
  RCA Persistence                  0.2448   0.6518   0.1331   0.1369   0.3359   0.1957   0.1150   0.0661
  Density                          0.0549   0.7782   0.1397   0.1274   0.8865   0.1088   0.1270   0.0670
  ECI                              0.0165   0.4821   0.0195   0.0248   0.4493   0.0348   0.0000      N/A
  ECI + Density                    0.0549   0.7782   0.1397   0.1274   0.8865   0.1088   0.1270   0.0670
  KNN (LLM embeddings)             0.0349   0.6462   0.0825   0.0761   0.6819   0.0736   0.1240   0.0353
  XGBoost                          0.2276   0.9020   0.3585   0.3235   0.9575   0.3130   0.5190   0.2389
  GNN-4F                           0.0727   0.8180   0.1048   0.1011   0.9141   0.1356  

## Cross-Year Consistency Check

How stable are the rankings across t=2015 and t=2016?

In [59]:
methods = [m for m in METHOD_ORDER if m in RESULTS[2015] and m in RESULTS[2016]]

for met in ['PR-AUC', 'AUROC', 'NDCG@20', 'CWR']:
    print(f'\n{met}:')
    print(f'  {"Method":<26} {"2015":>7} {"2016":>7} {"Delta":>7}')
    print(f'  {"-"*50}')
    for m in methods:
        v15 = RESULTS[2015][m].get(met, float('nan'))
        v16 = RESULTS[2016][m].get(met, float('nan'))
        if isinstance(v15, str) or isinstance(v16, str):
            print(f'  {m:<26} {"N/A":>7} {"N/A":>7} {"N/A":>7}')
        else:
            delta = v16 - v15
            print(f'  {m:<26} {v15:>7.4f} {v16:>7.4f} {delta:>+7.4f}')

# Save combined summary
rows = []
for m in methods:
    for yr in [2015, 2016]:
        r = RESULTS[yr].get(m, {})
        row = {'Method': m, 'Year': yr}
        row.update(r)
        rows.append(row)
df_all = pd.DataFrame(rows)
combined_path = os.path.join(OUT_DIR, 'full_universe_combined_results.csv')
df_all.to_csv(combined_path, index=False)
print(f'\nCombined results saved -> {combined_path}')


PR-AUC:
  Method                        2015    2016   Delta
  --------------------------------------------------
  RCA Persistence             0.2448  0.2470 +0.0022
  Density                     0.0549  0.0596 +0.0047
  ECI                         0.0165  0.0193 +0.0028
  ECI + Density               0.0549  0.0596 +0.0047
  KNN (LLM embeddings)        0.0349  0.0347 -0.0002
  XGBoost                     0.2276  0.2234 -0.0042
  GNN-4F                      0.0727  0.0764 +0.0037
  GNN-11F (BACI+WDI)          0.0811  0.0816 +0.0005
  GNN-11F+LLM                 0.0846  0.0837 -0.0009
  GNN-LLM v2 (GAT+Focal)      0.0741  0.0730 -0.0011
  GNN-LLM v2 Unopt            0.0473  0.0491 +0.0018
  GNN-LLM PCA-A (SAGE)        0.0877  0.0906 +0.0029
  GNN-LLM PCA-B (GCN+EW)      0.0914  0.0897 -0.0017
  GNN-LLM PCA-D (GAT+EW opt)  0.0746  0.0725 -0.0021

AUROC:
  Method                        2015    2016   Delta
  --------------------------------------------------
  RCA Persistence            

---

## RCA > 0.25 Filtered Evaluation

Re-evaluates all 8 methods on the subset of full-universe pairs where the country already has
**RCA[t] > 0.25** in that product at the observation year — i.e., the country has *some* existing
export activity in the product but has not yet crossed the smoothed RCA ≥ 1 threshold (M[t]=0).

This answers: *among products where a country is already partially competitive, which method best
identifies those that will gain full comparative advantage in 5 years?*

| Subset | Pairs (2015) | Positive rate (2015) |
|--------|-------------|----------------------|
| Full universe (M[t]=0) | 1,078,236 | 1.71% |
| **RCA > 0.25 filtered** | **~103,768** | **~11.23%** |

Scores are taken directly from the cached `SCORES[year][method]` arrays (no re-inference needed).
PCI weights are recomputed on the filtered df to preserve correct CWR normalisation.

In [53]:
def build_rca025_filtered(t, df_full):
    """
    Filter the full-universe df to pairs where raw RCA[t] > 0.25.
    Recomputes PCI weights on the filtered subset (min_pci recalculated
    so CWR normalisation is consistent within the filtered pool).
    Returns a reset-index DataFrame with an 'orig_idx' column pointing
    back to df_full's positional index (for score alignment).
    """
    rca_t = rca_df[rca_df['year'] == t][['country', 'product', 'rca']]
    df = df_full.copy()
    df = df.merge(rca_t, on=['country', 'product'], how='left')
    df['rca'] = df['rca'].fillna(0.0)
    df = df[df['rca'] > 0.25].copy()
    df['orig_idx'] = df.index   # positional index into df_full (for score lookup)

    # Recompute PCI weights on filtered subset only
    df['pci'] = df['product'].map(pci_dict).fillna(pci_fill_val)
    min_pci_filt = df['pci'].min()
    df['w'] = df['pci'] - min_pci_filt

    df = df.reset_index(drop=True)
    pos_rate = df['label'].mean() * 100
    print(f't={t} | RCA>0.25 filter: {len(df):,} pairs | '
          f'{df["label"].sum():,} positives ({pos_rate:.2f}%) | '
          f'{(df["label"]==0).sum():,} negatives')
    return df

df_2015_filt = build_rca025_filtered(2015, df_2015)
df_2016_filt = build_rca025_filtered(2016, df_2016)

t=2015 | RCA>0.25 filter: 103,768 pairs | 11,648 positives (11.23%) | 92,120 negatives
t=2016 | RCA>0.25 filter: 104,337 pairs | 11,961 positives (11.46%) | 92,376 negatives


In [60]:
RESULTS_FILT = {2015: {}, 2016: {}}

def evaluate_filtered(year, name, df_filt, skip_map10=False):
    full_scores = SCORES[year].get(name)
    if full_scores is None:
        print(f'  [SKIP] No cached scores for {name} year={year}')
        return None
    filt_scores = full_scores[df_filt['orig_idx'].values]
    res = compute_metrics(filt_scores, df_filt, skip_map10=skip_map10)
    RESULTS_FILT[year][name] = res
    m10s = res['mAP@10'] if skip_map10 else f"{res['mAP@10']:.4f}"
    print(f'[{year}|RCA>0.25] {name:<30}  PR-AUC={res["PR-AUC"]:.4f}  '
          f'NDCG@20={res["NDCG@20"]:.4f}  Prec@20={res["Prec@20"]:.4f}  '
          f'CWR={res["CWR"]:.4f}  BestF1={res["Best F1"]:.4f}  '
          f'P@1000={res["P@1000"]:.4f}  mAP@10={m10s}')
    return res

print('=== t=2015 — RCA > 0.25 filtered ===')
for name in METHOD_ORDER:
    skip = (name == 'ECI')
    evaluate_filtered(2015, name, df_2015_filt, skip_map10=skip)

print()
print('=== t=2016 — RCA > 0.25 filtered ===')
for name in METHOD_ORDER:
    skip = (name == 'ECI')
    evaluate_filtered(2016, name, df_2016_filt, skip_map10=skip)

=== t=2015 — RCA > 0.25 filtered ===
[2015|RCA>0.25] RCA Persistence                 PR-AUC=0.3681  NDCG@20=0.2186  Prec@20=0.1891  CWR=0.4717  BestF1=0.2849  P@1000=0.1930  mAP@10=0.1186
[2015|RCA>0.25] Density                         PR-AUC=0.1278  NDCG@20=0.3081  Prec@20=0.2275  CWR=0.5664  BestF1=0.2141  P@1000=0.1600  mAP@10=0.1669
[2015|RCA>0.25] ECI                             PR-AUC=0.1083  NDCG@20=0.1342  Prec@20=0.1387  CWR=0.4875  BestF1=0.2018  P@1000=0.0930  mAP@10=N/A
[2015|RCA>0.25] ECI + Density                   PR-AUC=0.1278  NDCG@20=0.3081  Prec@20=0.2275  CWR=0.5664  BestF1=0.2141  P@1000=0.1600  mAP@10=0.1669
[2015|RCA>0.25] KNN (LLM embeddings)            PR-AUC=0.1508  NDCG@20=0.2326  Prec@20=0.1770  CWR=0.5874  BestF1=0.2183  P@1000=0.2900  mAP@10=0.1114
[2015|RCA>0.25] XGBoost                         PR-AUC=0.3087  NDCG@20=0.4283  Prec@20=0.3327  CWR=0.8313  BestF1=0.3729  P@1000=0.5180  mAP@10=0.2786
[2015|RCA>0.25] GNN-4F                          PR-AUC=0.147

In [61]:
METRICS = ['PR-AUC', 'AUROC', 'NDCG@20', 'Prec@20', 'CWR', 'Best F1', 'P@1000', 'mAP@10']

for yr, df_filt in [(2015, df_2015_filt), (2016, df_2016_filt)]:
    n_pairs  = len(df_filt)
    pos_rate = df_filt['label'].mean() * 100
    methods  = [m for m in METHOD_ORDER if m in RESULTS_FILT[yr]]

    print(f'\n{"="*102}')
    print(f'  RCA > 0.25 FILTERED EVALUATION — t={yr}  ({n_pairs:,} pairs, {pos_rate:.2f}% positive)')
    print(f'{"="*102}')
    print(f'  {"Method":<30}' + ''.join(f'{m:>9}' for m in METRICS))
    print(f'  {"-"*102}')

    rows = []
    for m in methods:
        r = RESULTS_FILT[yr][m]
        tag = ' <--' if 'GNN' in m else ''
        row_dict = {'Method': m}
        vals = []
        for met in METRICS:
            v = r.get(met, 'N/A')
            row_dict[met] = v
            vals.append(f'{"N/A":>9}' if v == 'N/A' else f'{v:>9.4f}')
        print(f'  {m:<30}' + ''.join(vals) + tag)
        rows.append(row_dict)

    print(f'{"="*102}')
    print('  N/A = ECI has no product-level ranking signal.')
    print('  Only pairs with RCA[t] > 0.25 at observation year included.')
    print('  CWR weights recomputed on filtered subset (min_pci recalculated).')

    df_res = pd.DataFrame(rows).set_index('Method')
    csv_path = os.path.join(OUT_DIR, f'full_universe_{yr}_rca025_results.csv')
    df_res.to_csv(csv_path)
    print(f'\n  Saved -> {csv_path}')

print('\nAll RCA > 0.25 filtered results saved.')


  RCA > 0.25 FILTERED EVALUATION — t=2015  (103,768 pairs, 11.23% positive)
  Method                           PR-AUC    AUROC  NDCG@20  Prec@20      CWR  Best F1   P@1000   mAP@10
  ------------------------------------------------------------------------------------------------------
  RCA Persistence                  0.3681   0.6197   0.2186   0.1891   0.4717   0.2849   0.1930   0.1186
  Density                          0.1278   0.5509   0.3081   0.2275   0.5664   0.2141   0.1600   0.1669
  ECI                              0.1083   0.4859   0.1342   0.1387   0.4875   0.2018   0.0930      N/A
  ECI + Density                    0.1278   0.5509   0.3081   0.2275   0.5664   0.2141   0.1600   0.1669
  KNN (LLM embeddings)             0.1508   0.5772   0.2326   0.1770   0.5874   0.2183   0.2900   0.1114
  XGBoost                          0.3087   0.7743   0.4283   0.3327   0.8313   0.3729   0.5180   0.2786
  GNN-4F                           0.1473   0.5979   0.2476   0.1795   0.6022   0.2

In [62]:
methods_filt = [m for m in METHOD_ORDER if m in RESULTS_FILT[2015] and m in RESULTS_FILT[2016]]

print('Cross-year consistency — RCA > 0.25 filtered\n')
for met in ['PR-AUC', 'AUROC', 'NDCG@20', 'CWR']:
    print(f'{met}:')
    print(f'  {"Method":<26} {"2015":>7} {"2016":>7} {"Delta":>7}')
    print(f'  {"-"*50}')
    for m in methods_filt:
        v15 = RESULTS_FILT[2015][m].get(met, float('nan'))
        v16 = RESULTS_FILT[2016][m].get(met, float('nan'))
        if isinstance(v15, str) or isinstance(v16, str):
            print(f'  {m:<26} {"N/A":>7} {"N/A":>7} {"N/A":>7}')
        else:
            delta = v16 - v15
            print(f'  {m:<26} {v15:>7.4f} {v16:>7.4f} {delta:>+7.4f}')
    print()

# Save combined filtered results
rows = []
for m in methods_filt:
    for yr in [2015, 2016]:
        r = RESULTS_FILT[yr].get(m, {})
        row = {'Method': m, 'Year': yr}
        row.update(r)
        rows.append(row)
df_filt_all = pd.DataFrame(rows)
combined_filt_path = os.path.join(OUT_DIR, 'full_universe_rca025_combined_results.csv')
df_filt_all.to_csv(combined_filt_path, index=False)
print(f'Combined RCA>0.25 filtered results saved -> {combined_filt_path}')

Cross-year consistency — RCA > 0.25 filtered

PR-AUC:
  Method                        2015    2016   Delta
  --------------------------------------------------
  RCA Persistence             0.3681  0.3641 -0.0040
  Density                     0.1278  0.1374 +0.0096
  ECI                         0.1083  0.1158 +0.0075
  ECI + Density               0.1278  0.1374 +0.0096
  KNN (LLM embeddings)        0.1508  0.1494 -0.0014
  XGBoost                     0.3087  0.3010 -0.0077
  GNN-4F                      0.1473  0.1551 +0.0078
  GNN-11F (BACI+WDI)          0.1670  0.1708 +0.0038
  GNN-11F+LLM                 0.1713  0.1773 +0.0060
  GNN-LLM v2 (GAT+Focal)      0.1622  0.1648 +0.0026
  GNN-LLM v2 Unopt            0.1170  0.1220 +0.0050
  GNN-LLM PCA-A (SAGE)        0.1676  0.1751 +0.0075
  GNN-LLM PCA-B (GCN+EW)      0.1745  0.1765 +0.0020
  GNN-LLM PCA-D (GAT+EW opt)  0.1629  0.1660 +0.0031

AUROC:
  Method                        2015    2016   Delta
  -----------------------------------

## Save All Results

Saves a combined (both years) CSV for the full-universe results alongside the per-year CSVs already written above.

In [63]:
METRICS = ['PR-AUC', 'AUROC', 'NDCG@20', 'Prec@20', 'CWR', 'Best F1', 'P@1000', 'mAP@10']

# ── Full-universe combined (all methods, both years) ──────────────────────────
fu_rows = []
for yr in [2015, 2016]:
    for m in METHOD_ORDER:
        r = RESULTS[yr].get(m)
        if r is None:
            continue
        row = {'Year': yr, 'Method': m}
        row.update({met: r.get(met, float('nan')) for met in METRICS})
        fu_rows.append(row)

df_fu_combined = pd.DataFrame(fu_rows)
combined_path = os.path.join(OUT_DIR, 'full_universe_combined_results.csv')
df_fu_combined.to_csv(combined_path, index=False)
print(f'Saved -> {combined_path}')

# ── RCA>0.25 filtered combined (all methods, both years) ──────────────────────
filt_rows = []
for yr in [2015, 2016]:
    for m in METHOD_ORDER:
        r = RESULTS_FILT[yr].get(m)
        if r is None:
            continue
        row = {'Year': yr, 'Method': m}
        row.update({met: r.get(met, float('nan')) for met in METRICS})
        filt_rows.append(row)

df_filt_combined = pd.DataFrame(filt_rows)
filt_combined_path = os.path.join(OUT_DIR, 'full_universe_rca025_combined_results.csv')
df_filt_combined.to_csv(filt_combined_path, index=False)
print(f'Saved -> {filt_combined_path}')

print(f'All results saved to {OUT_DIR}/')
import os as _os
for f in sorted(_os.listdir(OUT_DIR)):
    print(f'  {f}')


Saved -> full_universe_eval\full_universe_combined_results.csv
Saved -> full_universe_eval\full_universe_rca025_combined_results.csv
All results saved to full_universe_eval/
  full_universe_2015_rca025_results.csv
  full_universe_2015_results.csv
  full_universe_2016_rca025_results.csv
  full_universe_2016_results.csv
  full_universe_combined_results.csv
  full_universe_rca025_combined_results.csv


---
## Plots

Generates and saves 4 figures to :
- **Fig 1** — Grouped bar: 4 primary metrics, 2015 vs 2016 side-by-side
- **Fig 2** — Z-score heatmap: all 8 metrics × all methods (avg of both years)
- **Fig 3** — Cross-year delta bars: how much each method shifts 2015→2016
- **Fig 4** — Radar chart: trade-off profiles for top representative methods

In [ ]:
%matplotlib inline
import subprocess, sys
result = subprocess.run(
    [sys.executable, 'ClaudeFiles/plot_results.py'],
    capture_output=True, text=True, cwd='.'
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-2000:])

# Display plots inline
from IPython.display import Image, display
import os
plot_dir = 'full_universe_eval/plots'
for fname in sorted(os.listdir(plot_dir)):
    if fname.endswith('.png'):
        print(f'
{fname}')
        display(Image(filename=os.path.join(plot_dir, fname), width=900))
